# Lab: Parent-Child & Summary-Based Multi-Vector RAG

In standard RAG, you embed the exact chunk of text you want the LLM to read. If the chunk is too big, the search fails. If the chunk is too small, the LLM lacks context to answer.

**Multi-Vector RAG solves this.**
In this lab, we will:
1. Split a document into large **Parent Chunks** (great for LLM context).
2. Split those Parents into small **Child Chunks** (great for semantic search).
3. Generate **Summaries** for each Parent (also great for semantic search).
4. Store the Children and Summaries in **Qdrant** as vectors.
5. Store the Parents in a standard document store.

When you ask a question, Qdrant finds the best Child or Summary, but the retriever intercepts it and returns the massive Parent document to the LLM to generate the final answer.

In [72]:
!pip install -qU langchain langchain-classic langchain-community langchain-openai langchain-huggingface langchain-qdrant qdrant-client pypdf tiktoken flask numpy scipy scikit-learn sentence-transformers ipython


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports 

In [73]:
import uuid
import requests
from pypdf import PdfReader
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Retrieval and Storage
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.stores import InMemoryByteStore

# Vector Store
from langchain_qdrant import QdrantVectorStore

# Core Components
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from IPython.display import Markdown, display

### Step 2: Configure Models

In [ ]:
# Initialize Chat Model
llm = ChatOpenAI(
    openai_api_key="your-api-key",
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="openai/gpt-oss-20b:free",
    temperature=0.0
)

# Initialize Dense Embedding Model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Step 3: Initialize Storage Architecture or Use Existing if already made

In [ ]:
# Qdrant Cloud holds the child & summary VECTORS (the semantic search index)
vectorstore = QdrantVectorStore.from_texts(
    texts=["Initialize"], 
    embedding=embeddings, 
    url="your-endpoint",
    api_key="your-api-key",
    collection_name="multi_vector_collection"
)

# Will hold the full parent documents, keyed by doc_id, once the retriever wraps it
store = InMemoryByteStore()
id_key = "doc_id"

In [ ]:
# Use this instead of the cell above ONLY if you already ran this notebook before and the vectors already exist in Qdrant. Comment out the cell above and uncomment this one.
# vectorstore = QdrantVectorStore.from_existing_collection(
#     embedding=embeddings,
#     url="your-endpoint",
#     api_key="your-api-key",
#     collection_name="multi_vector_collection"
# )
#
# store = InMemoryByteStore()
# id_key = "doc_id"

### Step 4: Download & Extract Document

In [76]:
# Attention is All You Need paper (source document for this lab)
PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"
PDF_FILENAME = "attention_paper.pdf"

# Download the PDF and save it locally; raise an error on HTTP failure
response = requests.get(PDF_URL)
response.raise_for_status()

with open(PDF_FILENAME, "wb") as f:
    f.write(response.content)

In [77]:
# Extract text from every page and join pages with newlines into one string
reader = PdfReader(PDF_FILENAME)
raw_text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

print(f"Extracted {len(raw_text)} characters.")

Extracted 39611 characters.


### Step 5: Generate Parent Documents

In [78]:
# Parent chunks are LARGE (~10k chars) so the LLM has full context to answer from
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=200)
parent_chunks = parent_splitter.split_text(raw_text)

# Wrap each chunk in a Document (parents are stored, not embedded directly)
parent_docs = [Document(page_content=chunk) for chunk in parent_chunks]

print(f"Created {len(parent_docs)} Parent Documents.")

Created 5 Parent Documents.


### Step 6: Generate Child Documents

In [79]:
# Child chunks are SMALL (~400 chars) so they embed precisely for semantic search
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
child_docs = []

import hashlib

def stable_id(text):
    """Deterministic ID derived from the chunk text, so the same chunk always gets the same ID across runs."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

for doc in parent_docs:
    # Give each parent a stable ID, then tag every child with that same ID
    _id = stable_id(doc.page_content)
    doc.metadata["doc_id"] = _id
    
    child_splits = child_splitter.split_documents([doc])
    for child in child_splits:
        child.metadata["doc_id"] = _id
    
    child_docs.extend(child_splits)

print(f"Created {len(child_docs)} Child Documents.")

Created 117 Child Documents.


### Step 7: Define and Execute Summarization Pipeline

In [80]:
# Prompt: compress a parent chunk into a short summary while keeping key terms
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following core concepts concisely. Do not lose technical keywords.\n\nText: {context}"
)

# LCEL chain: pass text through unchanged → prompt → LLM → plain string output
summary_chain = (
    {"context": RunnablePassthrough()} 
    | summary_prompt 
    | llm 
    | StrOutputParser()
)

In [81]:
summary_docs = []

# Summarize each parent and carry its doc_id so the summary maps back to it
for doc in parent_docs:
    summary_text = summary_chain.invoke(doc.page_content)
    
    summary_doc = Document(
        page_content=summary_text,
        metadata={"doc_id": doc.metadata["doc_id"]}
    )
    summary_docs.append(summary_doc)

print(f"Generated {len(summary_docs)} Summaries.")

Generated 5 Summaries.


### Step 8: Configure Multi-Vector Retriever

In [82]:
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,   # searches child/summary embeddings
    byte_store=store,          # returns the linked parent documents
    id_key=id_key,             # the metadata field linking vectors to parents
    search_kwargs={"k": 2}    # return top-2 matches per query
)

# Store the full parent Documents in the docstore (auto-serialized into the ByteStore)
retriever.docstore.mset([(doc.metadata[id_key], doc) for doc in parent_docs])

# First-time setup: uploads vectors to Qdrant. If you already ran this notebook before and data already exists in Qdrant, comment out these two lines to avoid uploading duplicates.
retriever.vectorstore.add_documents(child_docs)
retriever.vectorstore.add_documents(summary_docs)

print("Multi-vector search index ready.")

Multi-vector search index ready.


### Step 9: Define the RAG Execution Function with Explainability

In [83]:
# Prompt instructing the LLM to answer AND explain its own reasoning/source usage
qa_template = """
You are a technical assistant. Answer the question using ONLY the provided context.

Context:
{context}

Question: {question}

Respond in exactly this format:

### Final Answer
<a clear, direct answer to the question>

### AI Tracing & Explainability
<Explain step by step how you arrived at the answer. Refer to the sources you used by their label (e.g. "Source 1"), state what each one contributed, and why it was relevant. Do not copy large blocks of text from the context — explain in your own words. Use as many lines as you need.>
"""

qa_prompt = ChatPromptTemplate.from_template(qa_template)

def format_docs(docs):
    """Label each source so the LLM can reference it by number in its tracing."""
    formatted = []
    for i, doc in enumerate(docs):
        parent_id = doc.metadata.get('doc_id', 'N/A')
        formatted.append(f"[Source {i+1}] (Parent ID: {parent_id})\n{doc.page_content}")
    return "\n\n".join(formatted)

# Run retrieval and capture the raw query in parallel
retrieval_chain = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
)

# Format the retrieved parents → prompt → LLM → answer text
generation_chain = (
    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Full pipeline: retrieval first, then generation on top of the retrieved context
rag_chain = retrieval_chain.assign(answer=generation_chain)

In [84]:
def run_rag_pipeline(query: str):
    """
    Executes the RAG pipeline and displays the LLM's own answer + reasoning.
    """
    response = rag_chain.invoke(query)
    display(Markdown(response["answer"]))
    return response

### Step 10: Execute Query

In [85]:
query = "What is the computational complexity per layer of a self-attention mechanism compared to a recurrent layer?"
response = run_rag_pipeline(query)

### Final Answer
A self‑attention layer has a per‑layer computational complexity of **O(n² · d)**, whereas a recurrent layer has a complexity of **O(n · d²)**. Thus, self‑attention scales quadratically with sequence length but linearly with representation dimension, while recurrent layers scale linearly with sequence length but quadratically with representation dimension.

### AI Tracing & Explainability
I extracted the complexity figures from Table 1 in the provided context. The table lists:

- **Self‑Attention**: complexity per layer = **O(n² · d)**.
- **Recurrent**: complexity per layer = **O(n · d²)**.

These entries directly answer the question by comparing the two mechanisms. No additional text was copied; I simply referenced the table’s values and explained the scaling relationship.